# TongDot Full Dictionary Fetcher

Fetches the COMPLETE TongDot dictionary using their autocomplete API.

## How It Works
1. **Get all words** from `/word/completion?term=` (returns ~27,784 words)
2. **Fetch definitions** for each word using the search page
3. **Multi-session support** for parallel speed (~6.5 w/s per session)

## Performance (27,784 words)
| Sessions | Combined Rate | Time  |
|----------|--------------|-------|
| 1        | 6.5 w/s      | ~1.2h |
| 2        | 13 w/s       | ~36m  |
| 3        | 19.5 w/s     | ~24m  |

## Required
Internet connection only — no external datasets needed.

In [ ]:
# Step 0: Setup
from __future__ import annotations

import json
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

def check_internet() -> bool:
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=5)
        return True
    except OSError:
        return False

if not check_internet():
    raise SystemError("No internet connection.")
print("Internet: OK")

try:
    import requests  # noqa: F401
    print("Packages: OK")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "requests", "-q"])
    print("Packages installed.")

IS_KAGGLE = Path("/kaggle").exists()
WORK_DIR = Path("/kaggle/working" if IS_KAGGLE else os.environ.get("ZOLOAI_WORK_DIR", ".")).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"IS_KAGGLE: {IS_KAGGLE}  |  WORK_DIR: {WORK_DIR}")


In [ ]:
# Step 1: Fetch ALL words from TongDot autocomplete API
import requests as req

COMPLETE_URL = "https://www.tongdot.com/word/completion"

print("Fetching complete word list from TongDot API...")
resp = req.get(COMPLETE_URL, params={"term": ""}, timeout=30)
resp.raise_for_status()
all_words = resp.json()

# Clean: strip whitespace, lowercase for dedup, remove empties
all_words = sorted(set(w.strip() for w in all_words if w.strip()))
print(f"Total unique words from API: {len(all_words):,}")
print(f"Sample: {all_words[:10]}")

# Save word list for reference
word_list_path = WORK_DIR / "tongdot_all_words.txt"
word_list_path.write_text("\n".join(all_words) + "\n", encoding="utf-8")
print(f"Word list saved: {word_list_path}")

# === MULTI-SESSION CONFIG ===
SESSION_ID = int(os.environ.get("TONGDOT_SESSION_ID", "0"))
NUM_SESSIONS = int(os.environ.get("TONGDOT_NUM_SESSIONS", "1"))
assert 0 <= SESSION_ID < NUM_SESSIONS

# Split words across sessions
words = all_words[SESSION_ID::NUM_SESSIONS]
print(f"\nSession {SESSION_ID + 1}/{NUM_SESSIONS}: {len(words):,} words")

# Config
MAX_WORKERS = 30
TIMEOUT = 15.0
RETRIES = 1
QUICK_TEST_N = 0  # Set > 0 to test with fewer words

if QUICK_TEST_N > 0:
    words = words[:QUICK_TEST_N]
    print(f"QUICK TEST MODE: {QUICK_TEST_N} words")

est_hours = len(words) / 6.5 / 3600
print(f"Estimated time: ~{est_hours*60:.1f} minutes")
if NUM_SESSIONS > 1:
    combined = len(all_words) / 6.5 / 3600 / NUM_SESSIONS
    print(f"All {NUM_SESSIONS} sessions: ~{combined*60:.1f} minutes")


In [ ]:
# Step 2: Fetcher logic — HTML parser + concurrent HTTP
import re
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from html import unescape
from html.parser import HTMLParser
from typing import Any
from urllib.parse import quote

BASE_SEARCH_URL = "https://www.tongdot.com/search/"
NO_RESULT_MARKER = "No word has been found in Tongdot.com Dictionary Database"
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/123.0.0.0 Safari/537.36"
)

OUTPUT_DIR = WORK_DIR / "tongdot_batches"
PROGRESS_LOG = WORK_DIR / "tongdot_fetch_progress.log"
STATUS_FILE = WORK_DIR / "tongdot_session_status.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _heartbeat(msg: str) -> None:
    line = f"{time.strftime('%H:%M:%S')}  [S{SESSION_ID}] {msg}\n"
    sys.stdout.write(line)
    sys.stdout.flush()
    with open(PROGRESS_LOG, "a", encoding="utf-8") as f:
        f.write(line)

def _write_status(status: str, written: int = 0, found: int = 0, errors: int = 0, rate: float = 0.0) -> None:
    try:
        data = json.loads(STATUS_FILE.read_text()) if STATUS_FILE.exists() else {}
        data[str(SESSION_ID)] = {"status": status, "written": written, "found": found, "errors": errors, "rate": round(rate, 1), "updated": time.strftime("%H:%M:%S")}
        STATUS_FILE.write_text(json.dumps(data, indent=2))
    except Exception:
        pass

def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", unescape(text)).strip()

class TongDotResultParser(HTMLParser):
    def __init__(self) -> None:
        super().__init__()
        self.results: list[dict[str, Any]] = []
        self._current: dict[str, Any] | None = None
        self._result_div_depth = 0
        self._inside_p = False
        self._current_p_parts: list[str] = []

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        attr_map = dict(attrs)
        classes = (attr_map.get("class") or "").split()
        if tag == "div" and "result" in classes:
            self._current = {"paragraphs": [], "audio_url": None}
            self._result_div_depth = 1
            self._inside_p = False
            self._current_p_parts = []
            return
        if self._current is None:
            return
        if tag == "div":
            self._result_div_depth += 1
        elif tag == "p":
            self._inside_p = True
            self._current_p_parts = []
        elif tag == "a" and not self._current.get("audio_url"):
            href = attr_map.get("href")
            if href:
                self._current["audio_url"] = href

    def handle_data(self, data: str) -> None:
        if self._current is not None and self._inside_p:
            self._current_p_parts.append(data)

    def handle_endtag(self, tag: str) -> None:
        if self._current is None:
            return
        if tag == "p" and self._inside_p:
            text = normalize_space("".join(self._current_p_parts))
            if text:
                self._current["paragraphs"].append(text)
            self._inside_p = False
            self._current_p_parts = []
            return
        if tag == "div":
            self._result_div_depth -= 1
            if self._result_div_depth == 0:
                self.results.append(self._finalize_result(self._current))
                self._current = None

    @staticmethod
    def _finalize_result(raw: dict[str, Any]) -> dict[str, Any]:
        paragraphs = raw.get("paragraphs", [])
        headword = re.sub(r":\s*$", "", paragraphs[0]).strip() if paragraphs else ""
        translation = paragraphs[1] if len(paragraphs) >= 2 else ""
        language = ""
        if len(paragraphs) >= 3:
            language = re.sub(r"^Language:\s*", "", paragraphs[2], flags=re.IGNORECASE).strip()
        return {
            "headword": headword, "translation": translation, "language": language,
            "audio_url": raw.get("audio_url"), "raw_paragraphs": paragraphs,
        }

def parse_results(html_text: str) -> list[dict[str, Any]]:
    parser = TongDotResultParser()
    parser.feed(html_text)
    parser.close()
    return [r for r in parser.results if r["headword"] or r["translation"] or r["language"]]

_thread_local = threading.local()

def _get_thread_session() -> req.Session:
    if not hasattr(_thread_local, "session"):
        s = req.Session()
        s.headers.update({"User-Agent": USER_AGENT})
        s.mount("https://", req.adapters.HTTPAdapter(
            pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS, pool_block=False
        ))
        _thread_local.session = s
    return _thread_local.session

def lookup_word(word: str) -> dict[str, Any]:
    source_url = BASE_SEARCH_URL + quote(word, safe="")
    session = _get_thread_session()
    resp = session.get(source_url, timeout=TIMEOUT)
    resp.raise_for_status()
    html_text = resp.text
    if NO_RESULT_MARKER in html_text:
        return {"query": word, "source_url": source_url, "found": False, "result_count": 0, "results": [], "error": None}
    results = parse_results(html_text)
    return {"query": word, "source_url": source_url, "found": bool(results), "result_count": len(results), "results": results, "error": None}

def load_processed_queries(output_path: Path) -> set[str]:
    processed: set[str] = set()
    if not output_path.exists():
        return processed
    with output_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            query = str(obj.get("query", "")).strip()
            if query:
                processed.add(query)
    return processed

_heartbeat("Fetcher functions loaded.")


In [ ]:
# Step 3: Process words with adaptive rate limiting

_rate_lock = threading.Lock()
_recent_429s: list[float] = []
_global_sleep = 0.0

def _adaptive_sleep() -> None:
    global _global_sleep
    with _rate_lock:
        now = time.time()
        _recent_429s[:] = [t for t in _recent_429s if now - t < 30]
        if len(_recent_429s) >= 3:
            _global_sleep = min(_global_sleep + 0.1, 2.0)
        elif len(_recent_429s) == 0 and _global_sleep > 0:
            _global_sleep = max(_global_sleep - 0.05, 0.0)
        sleep_time = _global_sleep
    if sleep_time > 0:
        time.sleep(sleep_time)

def _record_429() -> None:
    with _rate_lock:
        _recent_429s.append(time.time())

def process_words(word_list: list[str], output_path: Path) -> dict[str, Any]:
    processed = load_processed_queries(output_path)
    pending = [w for w in word_list if w not in processed]
    skipped = len(word_list) - len(pending)

    if not pending:
        _heartbeat(f"All {len(word_list)} words already processed. Skipping.")
        _write_status("skipped", skipped=skipped)
        return {"attempted": 0, "written": 0, "found": 0, "missed": 0, "errors": 0, "elapsed": 0, "skipped": skipped}

    written = found = missed = errors = 0
    started_at = time.time()
    error_types: dict[str, int] = {}
    write_lock = threading.Lock()
    batch_size = MAX_WORKERS * 3

    _write_status("running")
    mode = "a" if output_path.exists() else "w"
    with output_path.open(mode, encoding="utf-8") as out:
        for batch_start in range(0, len(pending), batch_size):
            batch = pending[batch_start:batch_start + batch_size]
            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                future_to_word = {executor.submit(_fetch_with_retry, w): w for w in batch}
                for future in as_completed(future_to_word):
                    word = future_to_word[future]
                    try:
                        record = future.result()
                    except Exception as exc:
                        errors += 1
                        error_types[type(exc).__name__] = error_types.get(type(exc).__name__, 0) + 1
                        record = {"query": word, "source_url": BASE_SEARCH_URL + quote(word, safe=""),
                                  "found": False, "result_count": 0, "results": [], "error": str(exc)}
                    if record["found"]: found += 1
                    else: missed += 1
                    with write_lock:
                        out.write(json.dumps(record, ensure_ascii=False) + "\n")
                        out.flush()
                        written += 1
                    if written % 100 == 0:
                        elapsed = time.time() - started_at
                        rate = written / elapsed if elapsed > 0 else 0
                        _heartbeat(f"{written + skipped}/{len(word_list):,} written={written} found={found} errors={errors} rate={rate:.1f}w/s")
                        _write_status("running", written=written, found=found, errors=errors, rate=rate)

    elapsed = time.time() - started_at
    rate = written / elapsed if elapsed > 0 else 0
    _write_status("done", written=written, found=found, errors=errors, rate=rate)
    _heartbeat(f"DONE: found={found}/{len(pending)} rate={rate:.1f}w/s in {elapsed:.0f}s")
    return {"attempted": len(pending), "written": written, "found": found,
            "missed": missed, "errors": errors, "elapsed": round(elapsed, 1),
            "skipped": skipped, "error_types": error_types}

def _fetch_with_retry(word: str) -> dict[str, Any]:
    _adaptive_sleep()
    last_exc = None
    for attempt in range(RETRIES + 1):
        try:
            return lookup_word(word)
        except req.exceptions.HTTPError as exc:
            last_exc = exc
            if exc.response is not None and exc.response.status_code == 429:
                _record_429()
                time.sleep(3.0 * (attempt + 1))
            elif attempt < RETRIES:
                time.sleep(1.0 * (attempt + 1))
            else:
                raise
        except Exception as exc:
            last_exc = exc
            if attempt < RETRIES:
                time.sleep(1.0 * (attempt + 1))
            else:
                raise
    raise RuntimeError(f"Fetch failed for {word}: {last_exc}")

_heartbeat("Processing functions defined.")


In [ ]:
# Step 4: Run the fetcher
SESSION_OUTPUT = OUTPUT_DIR / f"session_{SESSION_ID}.jsonl"

start_time = time.time()
_heartbeat(f"Starting session {SESSION_ID}/{NUM_SESSIONS}: {len(words):,} words")

try:
    summary = process_words(words, SESSION_OUTPUT)
    success = True
except Exception as e:
    _heartbeat(f"FAILED: {e}")
    _write_status("failed")
    summary = {"attempted": 0, "written": 0, "found": 0, "missed": 0, "errors": 0, "elapsed": 0, "error_types": {}}
    success = False

total_elapsed = time.time() - start_time
rate = summary.get("written", 0) / total_elapsed if total_elapsed > 0 else 0
_heartbeat(f"Session {SESSION_ID} complete in {total_elapsed/60:.1f} minutes ({rate:.1f} w/s)")

print(f"\n{'='*50}")
print(f"Session {SESSION_ID} Summary")
print(f"{'='*50}")
print(f"Words processed: {summary.get('written', 0):,}")
print(f"Found: {summary.get('found', 0):,}")
print(f"Missed: {summary.get('missed', 0):,}")
print(f"Errors: {summary.get('errors', 0):,}")
print(f"Rate: {rate:.1f} w/s")
print(f"Time: {total_elapsed/60:.1f} minutes")
print(f"Output: {SESSION_OUTPUT}")


In [ ]:
# Step 5: Merge all session outputs
FINAL_OUTPUT = WORK_DIR / "tongdot_dictionary.jsonl"

session_files = sorted(OUTPUT_DIR.glob("session_*.jsonl"))
print(f"Found {len(session_files)} session output(s)")
for f in session_files:
    with open(f) as fh:
        lines = sum(1 for _ in fh)
    print(f"  {f.name}: {lines:,} records")

if session_files:
    merged = found = errs = deduped = 0
    seen: set[str] = set()
    with open(FINAL_OUTPUT, "w", encoding="utf-8") as out:
        for sf in session_files:
            with open(sf, "r", encoding="utf-8") as inp:
                for line in inp:
                    ls = line.strip()
                    if not ls: continue
                    try:
                        obj = json.loads(ls)
                    except json.JSONDecodeError: continue
                    q = obj.get("query", "")
                    if q in seen:
                        deduped += 1
                        continue
                    seen.add(q)
                    out.write(ls + "\n")
                    merged += 1
                    if obj.get("found"): found += 1
                    if obj.get("error"): errs += 1

    print(f"\n{'='*50}")
    print(f"Merge Results")
    print(f"{'='*50}")
    print(f"Total records: {merged:,}")
    print(f"Found: {found:,} ({found/merged*100:.1f}%)" if merged else "Found: 0")
    print(f"Missed: {merged - found - errs:,}")
    print(f"Errors: {errs:,}")
    if deduped: print(f"Duplicates removed: {deduped}")
    print(f"File size: {FINAL_OUTPUT.stat().st_size/1024:.0f} KB")

    # Validate
    valid = True
    with open(FINAL_OUTPUT) as f:
        for i, line in enumerate(f, 1):
            try:
                obj = json.loads(line)
                if "query" not in obj:
                    print(f"Validation FAIL: line {i}")
                    valid = False; break
            except json.JSONDecodeError:
                print(f"Validation FAIL: line {i}")
                valid = False; break
    if valid: print("Validation: PASSED")

    s = {"source": "tongdot.com API", "output": str(FINAL_OUTPUT), "sessions": len(session_files),
         "total_records": merged, "found": found, "missed": merged - found - errs,
         "errors": errs, "duplicates_removed": deduped, "validation_passed": valid}
    summary_path = FINAL_OUTPUT.with_suffix(FINAL_OUTPUT.suffix + ".summary.json")
    summary_path.write_text(json.dumps(s, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Summary: {summary_path}")


In [ ]:
# Step 6: Monitor all sessions
if STATUS_FILE.exists():
    data = json.loads(STATUS_FILE.read_text())
    print(f"{'Session':<10} {'Status':<10} {'Written':>10} {'Found':>8} {'Rate':>10} {'Updated':>10}")
    print("-" * 62)
    tw = tf = 0
    for sid in sorted(data.keys(), key=int):
        s = data[sid]
        print(f"S{sid:<9} {s['status']:<10} {s.get('written',0):>10,} {s.get('found',0):>8,} {s.get('rate',0):>9.1f}w/s {s.get('updated','?'):>10}")
        tw += s.get("written", 0); tf += s.get("found", 0)
    print("-" * 62)
    print(f"{'Total':<10} {'':<10} {tw:>10,} {tf:>8,}")
    total_words = len(all_words) if 'all_words' in dir() else 27784
    print(f"Progress: {tw}/{total_words:,} ({tw/total_words*100:.1f}%)")
else:
    print("No status file yet.")


# Multi-Session Guide

## Quick Setup (3 sessions — ~24 minutes)
1. **Clone this notebook 3 times** in Kaggle ("Copy and Edit")
2. **In each clone**, change Step 1:
   ```python
   SESSION_ID = 0   # Clone 1: 0, Clone 2: 1, Clone 3: 2
   NUM_SESSIONS = 3  # Same in ALL clones
   ```
3. **Run all 3 notebooks** simultaneously
4. **Monitor** with Step 6, **merge** with Step 5

## Single Session (~1.2 hours)
Just run all cells as-is. No changes needed.

## Resuming
Each session auto-resumes from where it left off. Already-fetched words are skipped.